# Gender Coding Occupation

Statistics data from: https://www.bls.gov/cps/demographics/women-labor-force.htm
Table: Occupation, industry, and class of worker by sex and detailed occupation -> cpsaat11.xlsx

In [ ]:
import pandas as pd
from langchain_dartmouth.llms import ChatDartmouthCloud
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
occupation_gender = pd.read_excel(
    "../data/cpsaat11.xlsx",
    skiprows=8,
    usecols=[0, 2],
    names=["Occupation", "Percent Women"],
    na_values=["–"],
)
occupation_gender = occupation_gender.dropna()
occuation_to_gender = occupation_gender.set_index("Occupation")[
    "Percent Women"
].to_dict()

In [ ]:
occupations = occupation_gender.Occupation.unique().tolist()

In [ ]:
admissions = pd.read_csv("../data/derived/admissions.csv")
admissions_relevant_columns = [
    "Student ID",
    "Job 1 Organization",
    "Job #1 Industry Code",
    "Job 1 Title",
]

In [ ]:
interviews = pd.read_csv("../data/derived/interviews.csv")
interviews_relevant_columns = [
    "Student ID",
    "Employer",
    "Industry",
    "Job Function",
]

In [ ]:
outcomes = pd.read_csv("../data/derived/outcomes.csv")
outcomes_relevant_columns = [
    "Student ID",
    "Employer",
    "Detailed Function",
    "Detailed Industry",
]

In [ ]:
from langchain_core.output_parsers import JsonOutputParser


def get_census_occupation(student_record: pd.Series) -> dict:
    llm = ChatDartmouthCloud(
        model_name="google_genai.gemini-2.0-flash-001",
        max_tokens=1024,
    )
    occupation_prompt = ChatPromptTemplate(
        [
            (
                "system",
                "Your task is to standardize a collection of occupation titles to a fixed, pre-defined set "
                "of occupation titles. "
                "You will receive a record of student employment that already includes an occupation title. "
                "Map this title to the best-fitting one from the provided list of allowed titles. "
                "Discuss the provided data before responding with your "
                "final decision with a valid JSON with the following keys:\n"
                "- assessment\n"
                "- occupation\n"
                "If none of the provided options are a good fit, label it as N/A."
                "The available occupation titles are:\n\n{{occupation_titles}}.",
            ),
            ("human", "Here is the employment record: \n\n {{record}}"),
        ],
        template_format="jinja2",
    )
    occupation_mapper = occupation_prompt | llm | JsonOutputParser()

    return occupation_mapper.invoke(
        input={
            "occupation_titles": occupations,
            "record": student_record.to_json(),
        }
    )

In [ ]:
import json
from pathlib import Path
from joblib import Parallel, delayed
from tqdm.auto import tqdm


def process_single_record(idx, record, id_name, results_dir):
    """Process one record and save result immediately"""
    result_file = Path(results_dir) / f"result_{idx}.json"

    # Skip if already processed
    if result_file.exists():
        print(f"Skipping {idx} (already processed)", end="\r")
        return {"idx": idx, "status": "skipped"}

    try:
        response = get_census_occupation(record)
        response[id_name] = record[id_name]
        # Save immediately
        with open(result_file, "w") as f:
            json.dump({"idx": idx, "result": response}, f, indent=2)

        return {
            "idx": idx,
            "id": record[id_name],
            "status": "success",
            "result": response,
        }

    except Exception as e:
        # Save error info
        error_file = Path(results_dir) / f"error_{idx}.json"
        with open(error_file, "w") as f:
            json.dump({"idx": idx, "id": record[id_name], "error": str(e)}, f, indent=2)

        return {"idx": idx, "id": record[id_name], "status": "failed", "error": str(e)}


def process_parallel_with_saves(records, id_name, results_dir="results/occupations"):
    """Process in parallel with individual saves"""
    Path(results_dir).mkdir(exist_ok=True)

    records_data = [(idx, row) for idx, row in records.iterrows()]

    # Process in parallel, each saving its own result
    results = Parallel(n_jobs=-1)(
        delayed(process_single_record)(idx, record, id_name, results_dir)
        for idx, record in tqdm(records_data, desc="Processing records")
    )

    # Summarize results
    successful = [r for r in results if r["status"] == "success"]
    failed = [r for r in results if r["status"] == "failed"]
    skipped = [r for r in results if r["status"] == "skipped"]

    print(" " * 100, end="\r")
    print(f"✅ Successful: {len(successful)}")
    print(f"❌ Failed: {len(failed)}")
    print(f"⏭️ Skipped: {len(skipped)}")

    return results

## Process Admissions data

In [ ]:
results_dir = "results/occupations/admissions"
results = process_parallel_with_saves(
    admissions[admissions_relevant_columns],
    id_name="Student ID",
    results_dir=results_dir,
)

In [ ]:
n_bad = 0
records = []
for file in Path(results_dir).glob("*.json"):
    d = json.load(file.open())
    try:
        records.append(d["result"])
    except:
        print(d)
n_bad

In [ ]:
mapped_occupations = pd.DataFrame.from_records(records)
mapped_occupations

In [ ]:
admissions = admissions.merge(
    right=mapped_occupations.rename(columns={"occupation": "census_occupation"}),
).drop(columns="assessment")

In [ ]:
admissions["occupation_pct_women"] = admissions["census_occupation"].map(
    occuation_to_gender
)

## Process Interviews data

In [ ]:
results_dir = "results/occupations/interviews"
results = process_parallel_with_saves(
    interviews[interviews_relevant_columns],
    id_name=["Student ID"],
    results_dir=results_dir,
)

In [ ]:
n_bad = 0
records = []
for file in Path(results_dir).glob("*.json"):
    d = json.load(file.open())
    try:
        d["result"].update({"idx": d["idx"]})
        records.append(d["result"])
    except:
        print(d)
n_bad

In [ ]:
mapped_occupations = pd.DataFrame.from_records(records)
mapped_occupations.head()

In [ ]:
interviews = (
    interviews.reset_index(names="idx")
    .merge(
        right=mapped_occupations.rename(columns={"occupation": "census_occupation"}),
        on=["idx", "Student ID"],
    )
    .drop(columns=["assessment", "idx"])
)

interviews["occupation_pct_women"] = interviews["census_occupation"].map(
    occuation_to_gender
)

## Process outcomes data

In [ ]:
results_dir = "results/occupations/outcomes"
results = process_parallel_with_saves(
    outcomes[outcomes_relevant_columns],
    id_name="Student ID",
    results_dir=results_dir,
)

In [ ]:
n_bad = 0
records = []
for file in Path(results_dir).glob("*.json"):
    d = json.load(file.open())
    try:
        d["result"].update({"idx": d["idx"]})
        records.append(d["result"])
    except:
        print(d)
n_bad

In [ ]:
mapped_occupations = pd.DataFrame.from_records(records)
mapped_occupations.head()

In [ ]:
mapped_occupations.rename(columns={"occupation": "census_occupation"})

In [ ]:
outcomes = (
    outcomes.reset_index(names="idx")
    .merge(
        right=mapped_occupations.rename(columns={"occupation": "census_occupation"}),
    )
    .drop(columns=["assessment", "idx"])
)

outcomes["occupation_pct_women"] = outcomes["census_occupation"].map(
    occuation_to_gender
)

In [ ]:
outcomes.head()

## Add industry gender coding

In [ ]:
company_gender_share = pd.read_excel(
    "../data/derived/company_gender_share_manually_supplemented.xlsx"
)
company_gender_share = company_gender_share.set_index("name")["women_pct"].to_dict()

In [ ]:
admissions["industry_pct_women"] = admissions["Job 1 Organization"].map(
    company_gender_share
)

In [ ]:
interviews["industry_pct_women"] = interviews["Employer"].map(company_gender_share)

In [ ]:
outcomes["industry_pct_women"] = outcomes["Employer"].map(company_gender_share)

## Save processed data

In [ ]:
admissions.to_excel("../data/derived/admissions_gendered.xlsx", index=False)
interviews.to_excel("../data/derived/interviews_gendered.xlsx", index=False)
outcomes.to_excel("../data/derived/outcomes_gendered.xlsx", index=False)